In [3]:
import json
from pathlib import Path

# Paths
baseline_path = Path("/root/autodl-tmp/code/Vision-SR1-main/evaluation/judgments/vision_intuitor_lora/checkpoint-1500-for-rl/v0.0_3b_intuitor_lora/global_step_60/mathverse.jsonl")
method_path = Path("/root/autodl-tmp/code/Vision-SR1-main/evaluation/judgments/vision_intuitor_lora/checkpoint-1500-for-rl/v3.5_3b_intuitor_lora/global_step_78/mathverse.jsonl")


def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON decode error in {path} at line {i}: {e}") from e
    return rows


def infer_correct(row: dict):
    """Infer correctness from common fields. Returns True/False/None."""
    # 1) Preferred field in your files
    if "judge_verdict" in row and isinstance(row["judge_verdict"], str):
        v = row["judge_verdict"].strip().lower()
        if v == "correct":
            return True
        if v == "incorrect":
            return False

    # 2) Exact answer match fallback
    if "extracted_answer" in row and "ground_truth" in row:
        ea = str(row["extracted_answer"]).strip()
        gt = str(row["ground_truth"]).strip()
        if ea and gt:
            return ea == gt

    # 3) Generic bool-like keys
    bool_like_keys = [
        "is_correct", "correct", "acc", "accuracy", "judge", "pass", "passed", "hit", "success"
    ]
    for key in bool_like_keys:
        if key in row:
            x = row[key]
            if isinstance(x, bool):
                return x
            if isinstance(x, (int, float)):
                return x > 0
            if isinstance(x, str):
                s = x.strip().lower()
                if s in {"true", "1", "yes", "y", "correct", "pass", "passed"}:
                    return True
                if s in {"false", "0", "no", "n", "incorrect", "wrong", "fail", "failed"}:
                    return False

    # 4) Score-like fields
    for key in ["score", "final_score", "reward", "overall", "judge_score", "original_score"]:
        if key in row and isinstance(row[key], (int, float)):
            return row[key] > 0

    return None


def pick_id_key(rows_a, rows_b):
    """Pick a shared key for alignment if possible; else fallback to line index."""
    candidate_keys = [
        "question_id", "problem_id", "id", "uid", "sample_id", "item_id", "index", "idx"
    ]
    if not rows_a or not rows_b:
        return None

    keys_a = set().union(*(r.keys() for r in rows_a[:200]))
    keys_b = set().union(*(r.keys() for r in rows_b[:200]))
    shared = keys_a & keys_b

    for key in candidate_keys:
        if key in shared:
            non_null_a = sum(1 for r in rows_a if r.get(key) is not None)
            non_null_b = sum(1 for r in rows_b if r.get(key) is not None)
            if non_null_a > 0 and non_null_b > 0:
                return key
    return None


baseline_rows = load_jsonl(baseline_path)
method_rows = load_jsonl(method_path)

id_key = pick_id_key(baseline_rows, method_rows)

pairs = []
if id_key is None:
    # Fallback: align by line order
    n = min(len(baseline_rows), len(method_rows))
    for i in range(n):
        pairs.append((i, baseline_rows[i], method_rows[i]))
    align_mode = "line_index"
else:
    base_map = {r[id_key]: r for r in baseline_rows if r.get(id_key) is not None}
    # keep method order, avoid O(n^2) next(...)
    for mr in method_rows:
        cid = mr.get(id_key)
        if cid in base_map:
            pairs.append((cid, base_map[cid], mr))
    align_mode = f"id_key={id_key}"

# Coverage sanity check
base_none = sum(infer_correct(r) is None for r in baseline_rows)
method_none = sum(infer_correct(r) is None for r in method_rows)

improved = []
for sid, b, m in pairs:
    b_ok = infer_correct(b)
    m_ok = infer_correct(m)
    if b_ok is False and m_ok is True:
        improved.append(sid)

print(f"Alignment mode: {align_mode}")
print(f"Baseline samples: {len(baseline_rows)} | Method samples: {len(method_rows)} | Paired: {len(pairs)}")
print(f"Infer coverage: baseline None={base_none}, method None={method_none}")
print(f"Improved cases (baseline wrong, method correct): {len(improved)}")
print("\n前20个题目序列:")
for x in improved[:20]:
    print(x)

# Keep for downstream analysis
improved_case_ids = improved
improved_case_ids[:20]

Alignment mode: line_index
Baseline samples: 3940 | Method samples: 3940 | Paired: 3940
Infer coverage: baseline None=0, method None=0
Improved cases (baseline wrong, method correct): 470

前20个题目序列:
1
7
11
15
21
22
28
33
41
43
52
56
58
64
67
70
74
80
95
100


[1, 7, 11, 15, 21, 22, 28, 33, 41, 43, 52, 56, 58, 64, 67, 70, 74, 80, 95, 100]

In [4]:
from IPython.display import display, Markdown, Image
from pathlib import Path
import os


def _collect_image_candidates(row: dict):
    """Try to collect image paths/urls from common keys."""
    candidates = []
    keys = [
        "image", "images", "image_path", "image_paths", "img", "img_path", "multi_modal_data", "question"
    ]

    def _walk(v):
        if isinstance(v, str):
            lv = v.lower()
            if any(ext in lv for ext in [".png", ".jpg", ".jpeg", ".webp", ".bmp", ".gif"]) or lv.startswith(
                ("http://", "https://")
            ):
                candidates.append(v)
        elif isinstance(v, dict):
            for vv in v.values():
                _walk(vv)
        elif isinstance(v, (list, tuple)):
            for vv in v:
                _walk(vv)

    for k in keys:
        if k in row:
            _walk(row[k])

    # de-duplicate while preserving order
    uniq = []
    seen = set()
    for x in candidates:
        if x not in seen:
            seen.add(x)
            uniq.append(x)
    return uniq


def _pick_text(row: dict):
    for k in ["prompt", "question", "problem", "query", "instruction", "input"]:
        if k in row and isinstance(row[k], str):
            return row[k]
    return "[No question text field found]"


def show_improved_case(idx: int):
    """
    idx: index into improved_case_ids (0-based)
    """
    if "improved_case_ids" not in globals() or "pairs" not in globals():
        raise RuntimeError("Please run Cell 1 first to generate improved_case_ids and pairs.")

    if idx < 0 or idx >= len(improved_case_ids):
        raise IndexError(f"idx={idx} out of range, valid range: [0, {len(improved_case_ids)-1}]")

    target_sid = improved_case_ids[idx]

    # Find corresponding pair
    matched = None
    for sid, b_row, m_row in pairs:
        if sid == target_sid:
            matched = (sid, b_row, m_row)
            break

    if matched is None:
        raise RuntimeError(f"Could not find pair for improved_case_ids[{idx}] = {target_sid}")

    sid, b_row, m_row = matched

    prompt_text = _pick_text(m_row)
    b_resp = b_row.get("response", "[No response field]")
    m_resp = m_row.get("response", "[No response field]")
    b_verdict = b_row.get("judge_verdict", "N/A")
    m_verdict = m_row.get("judge_verdict", "N/A")

    display(Markdown(f"## Case idx={idx}, sequence={sid}"))
    display(Markdown("### Question Text"))
    display(Markdown("```text\n" + prompt_text + "\n```"))

    # Try showing image(s) if paths/urls exist
    image_candidates = _collect_image_candidates(m_row)
    if not image_candidates:
        image_candidates = _collect_image_candidates(b_row)

    display(Markdown("### Question Image(s)"))
    if not image_candidates:
        print("No explicit image path/url found in this record.")
    else:
        shown = 0
        for p in image_candidates:
            # local file
            if isinstance(p, str) and os.path.exists(p):
                display(Markdown(f"- local: {p}"))
                display(Image(filename=p))
                shown += 1
            # http(s) url
            elif isinstance(p, str) and p.startswith(("http://", "https://")):
                display(Markdown(f"- url: {p}"))
                display(Image(url=p))
                shown += 1
            else:
                display(Markdown(f"- unresolved image reference: {p}"))

        if shown == 0:
            print("Found image-like references, but none could be directly rendered.")

    display(Markdown("### Baseline Response"))
    display(Markdown(f"Verdict: **{b_verdict}**"))
    display(Markdown("```text\n" + str(b_resp) + "\n```"))

    display(Markdown("### Our Method Response"))
    display(Markdown(f"Verdict: **{m_verdict}**"))
    display(Markdown("```text\n" + str(m_resp) + "\n```"))


# Example: specify idx here
idx = 0
show_improved_case(idx)

## Case idx=0, sequence=1

### Question Text

```text
system
You are a helpful assistant.
user
As shown in the figure, it is known that angle A = 80.0, angle B = 60.0, DE parallel BC, then the size of angle CED is ()
Choices:
A:40°
B:60°
C:120°
D:140° You FIRST think about the reasoning process as an internal monologue and then provide the final answer. The reasoning process MUST BE enclosed within <think> </think> tags. The final answer MUST BE put in \boxed{}.
assistant

```

### Question Image(s)

No explicit image path/url found in this record.


...

Verdict: **incorrect**

```text
<think>The problem states that DE is parallel to BC. Therefore, angle ADE is equal to angle A = 80 degrees. In triangle BDE, the sum of the angles is 180 degrees. So, angle BDE + angle B + angle DEB = 180. Since angle B = 60 degrees, we have angle BDE + 60 + angle DEB = 180. Therefore, angle BDE + angle DEB = 120. Since angle BDE = angle ADE = 80, we have 80 + angle DEB = 120. Thus, angle DEB = 120 - 80 = 40. Angle CED is equal to angle DEB because they are corresponding angles. Therefore, angle CED = 40 degrees.
</think>
\boxed{A}
```

### Our Method Response

Verdict: **correct**

```text
<think>The problem states that DE is parallel to BC. Therefore, angle ADE is equal to angle A, which is 80 degrees. In triangle DEC, the sum of the angles is 180 degrees. So, angle DEC + angle C + angle CED = 180 degrees. We need to find angle CED. To do this, we first need to find angle DEC. Since DE is parallel to BC, angle DEC and angle C are alternate interior angles, so angle DEC = angle C. Therefore, angle C + angle C + angle CED = 180 degrees, which means 2*angle C + angle CED = 180 degrees. We know that angle A = 80 degrees and angle B = 60 degrees. Therefore, angle C = 180 - 80 - 60 = 40 degrees. So, angle DEC = 40 degrees. Therefore, angle CED = 180 - 40 = 140 degrees.
</think>
\boxed{D}
```